# SafeBroker Risk Controls Demo

**Docker image**: `ml4t`

**Section Reference**: 25.7 (Operational readiness)

**Implementation Skills**:
- ml4t.live.safety: SafeBroker, LiveRiskConfig, RiskState, RiskLimitError
- ml4t.live.safety: VirtualPortfolio for shadow mode position tracking
- Kill switch, position limits, order limits, drawdown monitoring

**Key Learning**:
This notebook drives six of SafeBroker's risk controls into their
failure modes against a synthetic broker:

1. **Order Size Limits**: Max shares and max value per order
2. **Position Limits**: Max value, shares, and total exposure
3. **Rate Limiting**: Max orders per minute
4. **Asset Restrictions**: Allowed/blocked asset lists
5. **Kill Switch**: Emergency halt that persists across restarts
6. **Shadow Mode**: VirtualPortfolio tracks fills without touching the broker

Three further controls, duplicate-order filtering, price-deviation checks
and daily-loss monitoring, are available through `LiveRiskConfig` and are
not demonstrated here. `13_runtime_safety_showcase` drives the daily-loss
kill-switch trip and stale-data rejection.

**Why This Matters**:
Live trading can lose real money. These safeguards provide defense-in-depth
against bugs, API errors, fat-finger mistakes, and runaway strategies.

**Learning Objectives**
- See how each protection layer fails closed when the strategy asks for something unsafe.
- Distinguish order-level checks from portfolio-level checks and persistent kill switches.
- Read the risk-control output as an operational checklist, not as a library feature tour.

**Prerequisites**
- Review Chapter 25.7 on operational readiness and the broker wrappers introduced earlier in the chapter.
- Familiarity with shadow mode and why paper trading alone is not enough for release gating.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# SafeBroker 風險控制示範

**Docker 映像檔**：`ml4t`

**章節參考**：25.7（營運準備）

**實作技能**：
- ml4t.live.safety：SafeBroker、LiveRiskConfig、RiskState、RiskLimitError
- ml4t.live.safety：在影子模式下追蹤部位的 VirtualPortfolio
- Kill Switch、部位限額、訂單限額、回撤監控

**學習重點**：
本 Notebook 使用合成券商，讓 SafeBroker 的六項風險控制進入各自的失敗情境：

1. **訂單規模限額**：每筆訂單的最大股數與最大金額
2. **部位限額**：最大金額、股數及總曝險
3. **速率限制**：每分鐘最多訂單數
4. **資產限制**：允許與封鎖的資產清單
5. **Kill Switch**：重啟後仍維持生效的緊急停止機制
6. **影子模式**：VirtualPortfolio 追蹤成交，但不觸及券商

另外三項控制——重複訂單過濾、價格偏離檢查與每日損失監控——可透過 `LiveRiskConfig` 使用，但此處不示範。`13_runtime_safety_showcase` 會示範每日損失觸發 Kill Switch，以及拒絕過時資料。

**重要原因**：
實盤交易可能造成真實資金損失。這些防護措施可針對程式錯誤、API 錯誤、手動輸入失誤，以及失控策略提供多層防護。

**學習目標**
- 觀察策略提出不安全請求時，每一層防護如何以拒絕為預設處置。
- 區分訂單層級檢查、投資組合層級檢查，以及持久化的 Kill Switch。
- 將風險控制輸出當作操作檢查清單閱讀，而非函式庫功能導覽。

**先備知識**
- 複習第 25.7 節的營運準備，以及本章前面介紹的券商包裝器。
- 了解影子模式，以及為何僅靠模擬交易不足以作為上線門檻。
<!-- ML4T_ZH_END c0000-p000 -->


## Setup

The setup is intentionally minimal because the point of the notebook is to expose the risk engine's
decisions. A lightweight mock broker is enough to make the failure modes and guardrails visible.

<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 設定

設定刻意保持精簡，因為本 Notebook 的重點是呈現風險引擎的決策。輕量的模擬券商已足以清楚呈現失敗模式與防護界線。
<!-- ML4T_ZH_END c0001-p000 -->


In [1]:
"""SafeBroker Risk Controls Demo: exercise six controls plus shadow accounting."""

import logging
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from async_utils import run_async
from ml4t.backtest.types import Order, OrderSide, OrderStatus, OrderType, Position
from ml4t.live import LiveRiskConfig, RiskLimitError, SafeBroker, VirtualPortfolio
from ml4t.live.protocols import ExecutionCapability

from utils.paths import get_output_dir

print("[OK] ml4t.live risk controls imported")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)

[OK] ml4t.live risk controls imported


In [2]:
# Production defaults; papermill injects overrides for CI
RATE_LIMIT_PER_MINUTE = 3
STATE_DIR = get_output_dir(25, "safety_risk_demo") / "temporary_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
MANAGED_STATE_PATHS: list[Path] = []

Each risk-control scenario receives a unique state path so persistence is testable without shared files.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
每個風險控制情境都使用獨立的狀態路徑，以便測試持久化行為而不共用檔案。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
def _temporary_state_path(prefix: str = "nb10_state_") -> str:
    """Return one isolated state path and track its state and journal files."""
    state_path = STATE_DIR / f"{prefix}{len(MANAGED_STATE_PATHS) // 2 + 1:02d}.json"
    journal_path = state_path.with_name(f"{state_path.stem}-journal{state_path.suffix}l")
    for path in (state_path, journal_path):
        path.unlink(missing_ok=True)
        MANAGED_STATE_PATHS.append(path)
    return str(state_path)

## 1. Mock Broker for Testing

The query surface separates broker state from fill mechanics. Every blocked order can therefore be
attributed to the risk layer rather than connectivity or market-data noise.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 1. 用於測試的模擬券商

查詢介面將券商狀態與成交機制分開。因此，每筆被封鎖的訂單都能歸因於風險層，而非連線或市場資料雜訊。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
class MockBrokerQueries:
    """Read-only and connection methods required by `SafeBroker`."""

    async def connect(self) -> None:
        self._connected = True
        logger.info("MockBroker: Connected")

    async def disconnect(self) -> None:
        self._connected = False
        logger.info("MockBroker: Disconnected")

    async def is_connected_async(self) -> bool:
        return self._connected

    @property
    def execution_capabilities(self):
        return frozenset({ExecutionCapability.LIMIT})

    def assert_paper_trading(self) -> None:
        return None

    @property
    def positions(self) -> dict[str, Position]:
        return self._positions.copy()

    @property
    def pending_orders(self) -> list[Order]:
        return self._pending_orders.copy()

    def get_position(self, asset: str) -> Position | None:
        return self._positions.get(asset)

    async def get_positions_async(self) -> dict[str, Position]:
        return self._positions.copy()

    async def get_pending_orders_async(self) -> list[Order]:
        return self._pending_orders.copy()

    async def get_position_async(self, asset: str) -> Position | None:
        return self._positions.get(asset)

    async def get_account_value_async(self) -> float:
        position_value = sum(
            abs(p.quantity) * (p.current_price or p.entry_price) for p in self._positions.values()
        )
        return self._cash + position_value

    async def get_cash_async(self) -> float:
        return self._cash

Order construction is pure apart from the broker-local sequence number.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
除了券商內部的序號，訂單建構不會產生其他副作用。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
def make_filled_order(
    broker: Any,
    asset: str,
    quantity: int,
    side: OrderSide,
    order_type: OrderType,
    limit_price: float | None,
    stop_price: float | None,
) -> Order:
    """Construct one immediately filled synthetic order."""
    broker._order_counter += 1
    price = limit_price or 100.0
    return Order(
        asset=asset,
        side=side,
        quantity=quantity,
        order_type=order_type,
        limit_price=limit_price,
        stop_price=stop_price,
        order_id=f"MOCK-{broker._order_counter}",
        status=OrderStatus.FILLED,
        filled_quantity=quantity,
        filled_price=price,
        filled_at=datetime.now(UTC),
    )

Fill accounting updates position quantity and cash from the same signed transaction.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
成交入帳會依同一筆帶正負號的交易，同時更新部位數量與現金。
<!-- ML4T_ZH_END c0010-p000 -->


In [6]:
def apply_mock_fill(broker: Any, order: Order) -> None:
    """Apply one filled order to synthetic broker state."""
    asset, quantity, side = order.asset, order.quantity, order.side
    price = float(order.filled_price)
    signed_qty = quantity if side == OrderSide.BUY else -quantity
    current = broker._positions.get(asset)
    if current:
        new_qty = current.quantity + signed_qty
        if new_qty == 0:
            del broker._positions[asset]
        else:
            current.quantity = new_qty
    else:
        broker._positions[asset] = Position(
            asset=asset,
            quantity=signed_qty,
            entry_price=price,
            entry_time=datetime.now(UTC),
            current_price=price,
        )
    transaction = quantity * price
    broker._cash += transaction if side == OrderSide.SELL else -transaction

The mutable broker adds immediate-fill submission to the query surface.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
可變動狀態的券商在查詢介面之外，加入立即成交的送單功能。
<!-- ML4T_ZH_END c0012-p000 -->


In [7]:
class MockBroker(MockBrokerQueries):
    """Synthetic broker for exercising `SafeBroker` controls."""

    def __init__(self, initial_cash: float = 100_000.0):
        self._cash = initial_cash
        self._positions: dict[str, Position] = {}
        self._pending_orders: list[Order] = []
        self._connected = False
        self._order_counter = 0

    async def submit_order_async(
        self,
        asset: str,
        quantity: int,
        side: OrderSide | None = None,
        order_type: OrderType = OrderType.MARKET,
        limit_price: float | None = None,
        stop_price: float | None = None,
        **_: Any,
    ) -> Order:
        """Fill one order immediately and update synthetic account state."""
        if side is None:
            side = OrderSide.BUY if quantity > 0 else OrderSide.SELL
            quantity = abs(quantity)
        order = make_filled_order(self, asset, quantity, side, order_type, limit_price, stop_price)
        apply_mock_fill(self, order)
        logger.info("MockBroker: %s %s %s @ $%.2f", side.value, quantity, asset, order.filled_price)
        return order

    async def cancel_order_async(self, order_id: str) -> bool:
        return False

    async def close_position_async(self, asset: str) -> Order | None:
        pos = self._positions.get(asset)
        if pos and pos.quantity != 0:
            side = OrderSide.SELL if pos.quantity > 0 else OrderSide.BUY
            return await self.submit_order_async(asset, abs(pos.quantity), side)
        return None

## 2. Demo Helper Function

One helper runs every attempted order and prints its outcome, so the notebook reads like an
operator console: either the request is allowed, or the control layer explains why it is
blocked. Each expected allow or block becomes an executable assertion.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 2. 示範輔助函式

同一個輔助函式會執行每次訂單嘗試並列印結果，讓本 Notebook 讀起來像操作人員主控台：請求不是獲准，就是由控制層說明封鎖原因。每個預期的允許或封鎖結果，都會成為可執行的斷言。
<!-- ML4T_ZH_END c0014-p000 -->


In [8]:
def run_demo(coro, expect_error: bool = False):
    """Run an async demo and fail the notebook on an unexpected outcome."""
    try:
        result = run_async(coro)
    except RiskLimitError as e:
        if not expect_error:
            raise AssertionError(f"Unexpected risk block: {e}") from e
        print(f"   [OK] Blocked as expected: {e}")
        return None
    if expect_error:
        raise AssertionError("Expected RiskLimitError, but the order was accepted")
    print("   [OK] Order accepted by control layer")
    return result

Every demo below reports its outcome through the same helper, so an allowed order and a
blocked one are distinguishable at a glance. A production risk layer needs the same property:
inconsistent error reporting is what makes a real incident hard to diagnose under time pressure.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
以下每個示範都透過同一輔助函式報告結果，因此允許與封鎖的訂單一眼即可區分。正式環境的風險層也需要這種特性：錯誤回報不一致，會使真實事故難以在時間壓力下診斷。
<!-- ML4T_ZH_END c0016-p000 -->


## 3. Demonstration: Order Size Limits

SafeBroker enforces maximum order size (shares and value).

This first demo answers the simplest production question: can one buggy order wipe out the session before
anything else has a chance to react?

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 3. 示範：訂單規模限額

SafeBroker 強制執行訂單規模上限（股數與金額）。

第一個示範回答最基本的正式環境問題：一筆有錯誤的訂單，是否會在其他機制來得及反應前就耗盡整個交易時段的資源？
<!-- ML4T_ZH_END c0017-p000 -->


In [9]:
print("\n" + "=" * 70)
print("DEMO 1: Order Size Limits")
print("=" * 70)

# Use temp file for state to avoid conflicts
state_file = _temporary_state_path()

broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    execution_mode="paper",
    max_order_shares=50,  # Max 50 shares per order
    max_order_value=5_000.0,  # Max $5,000 per order
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
# Prime the staleness guard with reference quotes for every asset this demo touches.
safe_broker.record_market_snapshot("AAPL", 100.0)
safe_broker.record_market_snapshot("TSLA", 600.0)

print("\n1. Order within limits (10 shares @ $100 = $1,000):")
run_demo(safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY))

print("\n2. Order exceeds share limit (100 shares, limit is 50):")
run_demo(
    safe_broker.submit_order_async("AAPL", 100, OrderSide.BUY),
    expect_error=True,
)

print("\n3. Order exceeds value limit (10 shares @ $600 = $6,000):")
# First set a price by creating a position
broker._positions["TSLA"] = Position(
    asset="TSLA",
    quantity=1,
    entry_price=600.0,
    entry_time=datetime.now(UTC),
    current_price=600.0,
)
run_demo(
    safe_broker.submit_order_async(
        "TSLA", 10, OrderSide.BUY, order_type=OrderType.LIMIT, limit_price=600.0
    ),
    expect_error=True,
)


DEMO 1: Order Size Limits
2026-09-08 21:32:00,328 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,339 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



1. Order within limits (10 shares @ $100 = $1,000):
2026-09-08 21:32:00,361 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 AAPL


2026-09-08 21:32:00,361 - __main__ - INFO - MockBroker: buy 10.0 AAPL @ $100.00


   [OK] Order accepted by control layer

2. Order exceeds share limit (100 shares, limit is 50):
   [OK] Blocked as expected: Order quantity 100.0 exceeds max 50

3. Order exceeds value limit (10 shares @ $600 = $6,000):
   [OK] Blocked as expected: Order value $6,000 exceeds max $5,000


Order-size controls stop both an oversized share count and an oversized notional before the
broker sees the request, which is what keeps a single mistyped instruction too small to matter.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
訂單規模控制會在券商收到請求前，同時攔下超額股數與超額名目金額，使單筆輸入錯誤的指令不致造成重大影響。
<!-- ML4T_ZH_END c0019-p000 -->


## 4. Demonstration: Position Limits

SafeBroker enforces maximum position size and total exposure. This answers the portfolio-level question of
whether a sequence of valid orders can still push the account into an unsafe aggregate state.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
## 4. 示範：部位限額

SafeBroker 強制執行部位規模與總曝險上限。這回答投資組合層級的問題：一連串各自有效的訂單，是否仍可能讓帳戶的整體狀態變得不安全？
<!-- ML4T_ZH_END c0020-p000 -->


In [10]:
print("\n" + "=" * 70)
print("DEMO 2: Position Limits")
print("=" * 70)

# Isolate the share cap by setting the dollar limits well above the attempted position.
state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    execution_mode="paper",
    max_position_value=100_000.0,
    max_position_shares=100,
    max_total_exposure=200_000.0,
    max_order_value=20_000.0,
    max_order_shares=200,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
safe_broker.record_market_snapshot("AAPL", 100.0)

print("\n1. Share cap: start with 50 shares @ $100:")
run_demo(safe_broker.submit_order_async("AAPL", 50, OrderSide.BUY))
print("\n2. Share cap: adding 60 would reach 110 shares:")
run_demo(
    safe_broker.submit_order_async("AAPL", 60, OrderSide.BUY),
    expect_error=True,
)


DEMO 2: Position Limits
2026-09-08 21:32:00,408 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,420 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



1. Share cap: start with 50 shares @ $100:


2026-09-08 21:32:00,436 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 50.0 AAPL


2026-09-08 21:32:00,437 - __main__ - INFO - MockBroker: buy 50.0 AAPL @ $100.00


   [OK] Order accepted by control layer

2. Share cap: adding 60 would reach 110 shares:
   [OK] Blocked as expected: Position quantity at risk 110.0 would exceed max 100


A separate broker isolates the per-position dollar cap from the share and total-exposure gates.

<!-- ML4T_ZH_BEGIN c0022-p000 -->
使用另一個獨立券商，將個別部位的金額上限與股數、總曝險限制分開測試。
<!-- ML4T_ZH_END c0022-p000 -->


In [11]:
state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())
config = LiveRiskConfig(
    execution_mode="paper",
    max_position_value=10_000.0,
    max_position_shares=200,
    max_total_exposure=200_000.0,
    max_order_value=20_000.0,
    max_order_shares=200,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
safe_broker.record_market_snapshot("MSFT", 150.0)

print("\n3. Position-value cap: 100 MSFT @ $150 would be $15,000:")
run_demo(
    safe_broker.submit_order_async(
        "MSFT", 100, OrderSide.BUY, order_type=OrderType.LIMIT, limit_price=150.0
    ),
    expect_error=True,
)

2026-09-08 21:32:00,477 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,488 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



3. Position-value cap: 100 MSFT @ $150 would be $15,000:
   [OK] Blocked as expected: Position value $15,000 would exceed max $10,000


A third broker builds two valid positions before a new order crosses only the total-exposure cap.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
第三個券商先建立兩個有效部位，再測試一筆只會跨越總曝險上限的新訂單。
<!-- ML4T_ZH_END c0024-p000 -->


In [12]:
state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())
config = LiveRiskConfig(
    execution_mode="paper",
    max_position_value=20_000.0,
    max_position_shares=200,
    max_total_exposure=15_000.0,
    max_order_value=20_000.0,
    max_order_shares=200,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
for symbol in ("AAPL", "MSFT", "GOOGL"):
    safe_broker.record_market_snapshot(symbol, 100.0)

print("\n4. Total exposure: build valid $5,000 and $8,000 positions:")
run_demo(safe_broker.submit_order_async("AAPL", 50, OrderSide.BUY))
run_demo(safe_broker.submit_order_async("MSFT", 80, OrderSide.BUY))
print("\n5. Total exposure: another $3,000 would raise exposure to $16,000:")
run_demo(
    safe_broker.submit_order_async("GOOGL", 30, OrderSide.BUY),
    expect_error=True,
)

2026-09-08 21:32:00,503 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,519 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



4. Total exposure: build valid $5,000 and $8,000 positions:


2026-09-08 21:32:00,545 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 50.0 AAPL


2026-09-08 21:32:00,546 - __main__ - INFO - MockBroker: buy 50.0 AAPL @ $100.00


   [OK] Order accepted by control layer
2026-09-08 21:32:00,589 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 80.0 MSFT


2026-09-08 21:32:00,589 - __main__ - INFO - MockBroker: buy 80.0 MSFT @ $100.00


   [OK] Order accepted by control layer

5. Total exposure: another $3,000 would raise exposure to $16,000:
   [OK] Blocked as expected: Total exposure $16,000 would exceed max $15,000


Each scenario above uses its own broker so the share cap, the per-position value cap and the
total-exposure cap each raise on their own account, with no earlier control masking the
rejection under test. Portfolio-level caps exist because a failure often arrives as a sequence
of individually reasonable orders that add up to an unreasonable position.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
上述每個情境都使用自己的券商，讓股數上限、個別部位金額上限與總曝險上限各自獨立觸發，不會被較早執行的控制措施掩蓋待測的拒絕結果。投資組合層級限額之所以必要，是因為失敗往往來自一連串單看合理、加總後卻形成不合理部位的訂單。
<!-- ML4T_ZH_END c0026-p000 -->


## 5. Demonstration: Rate Limiting

SafeBroker limits the number of orders per minute. Rate limiting matters because repeated small errors can
be just as destructive as one oversized order when the loop is running unattended.

<!-- ML4T_ZH_BEGIN c0027-p000 -->
## 5. 示範：速率限制

SafeBroker 限制每分鐘的訂單數量。速率限制很重要，因為循環無人看管時，反覆發生的小錯誤可能和單筆超大訂單一樣具有破壞力。
<!-- ML4T_ZH_END c0027-p000 -->


In [13]:
print("\n" + "=" * 70)
print("DEMO 3: Rate Limiting")
print("=" * 70)

state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    execution_mode="paper",
    max_orders_per_minute=RATE_LIMIT_PER_MINUTE,
    max_order_value=50_000.0,
    max_position_value=100_000.0,
    dedup_window_seconds=0.0,  # Disable dedup for this demo
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)

print("\nSubmitting 5 orders rapidly (limit is 3/minute):")
symbols = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]  # Different symbols to avoid dedup
for sym in symbols:
    safe_broker.record_market_snapshot(sym, 100.0)
for i in range(5):
    print(f"\nOrder {i + 1} ({symbols[i]}):")
    run_demo(
        safe_broker.submit_order_async(symbols[i], 10, OrderSide.BUY),
        expect_error=(i >= 3),  # Expect error after 3rd order
    )


DEMO 3: Rate Limiting
2026-09-08 21:32:00,648 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,663 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



Submitting 5 orders rapidly (limit is 3/minute):

Order 1 (AAPL):


2026-09-08 21:32:00,693 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 AAPL


2026-09-08 21:32:00,693 - __main__ - INFO - MockBroker: buy 10.0 AAPL @ $100.00


   [OK] Order accepted by control layer

Order 2 (MSFT):
2026-09-08 21:32:00,741 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 MSFT


2026-09-08 21:32:00,742 - __main__ - INFO - MockBroker: buy 10.0 MSFT @ $100.00


   [OK] Order accepted by control layer

Order 3 (GOOGL):


2026-09-08 21:32:00,786 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 GOOGL


2026-09-08 21:32:00,786 - __main__ - INFO - MockBroker: buy 10.0 GOOGL @ $100.00


   [OK] Order accepted by control layer

Order 4 (AMZN):
   [OK] Blocked as expected: Rate limit: 3/min exceeded

Order 5 (META):
   [OK] Blocked as expected: Rate limit: 3/min exceeded


The burst is rejected on the fourth order even though every order in it passes every other
check. A throughput cap is what stands between the account and a runaway loop, a duplicated
signal, or a broker rate-limit ban earned by chatty execution code.

<!-- ML4T_ZH_BEGIN c0029-p000 -->
即使這組突發訂單中的每筆訂單都通過其他所有檢查，第四筆仍會遭到拒絕。吞吐量上限可保護帳戶，避免失控循環、重複訊號，或過度頻繁的執行程式碼導致券商因觸及速率限制而封鎖帳戶。
<!-- ML4T_ZH_END c0029-p000 -->


## 6. Demonstration: Asset Restrictions

SafeBroker can restrict trading to allowed assets or block specific assets. This is how a live system keeps
a strategy inside its approved mandate even if symbol selection logic goes wrong.

<!-- ML4T_ZH_BEGIN c0030-p000 -->
## 6. 示範：資產限制

SafeBroker 可將交易限制在允許的資產，或封鎖特定資產。即使標的選擇邏輯出錯，實盤系統仍可藉此讓策略維持在核准的交易授權範圍內。
<!-- ML4T_ZH_END c0030-p000 -->


In [14]:
print("\n" + "=" * 70)
print("DEMO 4: Asset Restrictions")
print("=" * 70)

state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

# Only allow specific ETFs
config = LiveRiskConfig(
    execution_mode="paper",
    allowed_assets={"SPY", "QQQ", "IWM"},  # Whitelist
    max_order_value=50_000.0,
    max_position_value=100_000.0,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
# Only prime SPY: AAPL is blocked at the asset-restriction check before staleness ever runs.
safe_broker.record_market_snapshot("SPY", 100.0)

print("\n1. Trading SPY (allowed):")
run_demo(safe_broker.submit_order_async("SPY", 10, OrderSide.BUY))

print("\n2. Trading AAPL (not in allowed list):")
run_demo(
    safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY),
    expect_error=True,
)


DEMO 4: Asset Restrictions
2026-09-08 21:32:00,834 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,846 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



1. Trading SPY (allowed):


2026-09-08 21:32:00,869 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 SPY


2026-09-08 21:32:00,870 - __main__ - INFO - MockBroker: buy 10.0 SPY @ $100.00


   [OK] Order accepted by control layer

2. Trading AAPL (not in allowed list):
   [OK] Blocked as expected: Asset AAPL not in allowed list


Now exercise the blocklist path: trades to non-blocked assets are
accepted, trades to any listed symbol are rejected before they reach
the broker.

<!-- ML4T_ZH_BEGIN c0032-p000 -->
接著測試封鎖清單路徑：未被封鎖的資產交易會獲准；清單上任何標的的交易，都會在到達券商前遭拒絕。
<!-- ML4T_ZH_END c0032-p000 -->


In [15]:
print("\n--- Testing Blocked Assets ---")
state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    execution_mode="paper",
    blocked_assets={"TSLA", "GME", "AMC"},  # Blacklist volatile stocks
    max_order_value=50_000.0,
    max_position_value=100_000.0,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
# Only prime AAPL: GME is blocked at the asset-restriction check before staleness runs.
safe_broker.record_market_snapshot("AAPL", 100.0)

print("\n3. Trading AAPL (not blocked):")
run_demo(safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY))

print("\n4. Trading GME (blocked):")
run_demo(
    safe_broker.submit_order_async("GME", 10, OrderSide.BUY),
    expect_error=True,
)


--- Testing Blocked Assets ---
2026-09-08 21:32:00,922 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:00,938 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



3. Trading AAPL (not blocked):


2026-09-08 21:32:00,961 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 AAPL


2026-09-08 21:32:00,961 - __main__ - INFO - MockBroker: buy 10.0 AAPL @ $100.00


   [OK] Order accepted by control layer

4. Trading GME (blocked):
   [OK] Blocked as expected: Asset GME is blocked


The allowlist and the blocklist put a hard boundary around what the strategy may touch. That
boundary is an operational safeguard rather than a research convenience: it is what stops a
symbol-selection bug from routing an order into an unsupported or explicitly banned instrument.

<!-- ML4T_ZH_BEGIN c0034-p000 -->
允許清單與封鎖清單為策略可觸及的標的設下硬性界線。這道界線是營運防護措施，而非研究上的便利設計：它能防止標的選擇錯誤將訂單送往不受支援或明確禁止的商品。
<!-- ML4T_ZH_END c0034-p000 -->


## 7. Demonstration: Kill Switch

The kill switch is an emergency halt that persists across restarts. It exists for the scenarios where the
safest action is to stop every new order until a human explicitly clears the system.

<!-- ML4T_ZH_BEGIN c0035-p000 -->
## 7. 示範：Kill Switch

Kill Switch 是重啟後仍持續生效的緊急停止機制。當最安全的做法是停止所有新訂單，直到有人明確解除系統停止狀態時，就需要它。
<!-- ML4T_ZH_END c0035-p000 -->


In [16]:
print("\n" + "=" * 70)
print("DEMO 5: Kill Switch")
print("=" * 70)

state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    execution_mode="paper",
    max_order_value=50_000.0,
    max_position_value=100_000.0,
    dedup_window_seconds=0.0,  # Disable for demo
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
safe_broker.record_market_snapshot("AAPL", 100.0)

print("\n1. Normal trading before kill switch:")
run_demo(safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY))

print("\n2. Activating kill switch (manual emergency halt):")
safe_broker.enable_kill_switch("Manual test - simulating emergency")
print("   Kill switch activated!")

print("\n3. Attempting to trade with kill switch active:")
run_demo(
    safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY),
    expect_error=True,
)


DEMO 5: Kill Switch
2026-09-08 21:32:01,008 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:01,019 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper



1. Normal trading before kill switch:


2026-09-08 21:32:01,041 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 AAPL


2026-09-08 21:32:01,041 - __main__ - INFO - MockBroker: buy 10.0 AAPL @ $100.00


   [OK] Order accepted by control layer

2. Activating kill switch (manual emergency halt):
2026-09-08 21:32:01,071 - ml4t.live.safety - CRITICAL - KILL SWITCH ACTIVATED: Manual test - simulating emergency


   Kill switch activated!

3. Attempting to trade with kill switch active:
   [OK] Blocked as expected: Kill switch active: Manual test - simulating emergency


Reconstruct `SafeBroker` from the same state file. The in-memory toggle is
irrelevant if the latch does not come back with the new instance.

<!-- ML4T_ZH_BEGIN c0037-p000 -->
從同一個狀態檔重建 `SafeBroker`。若鎖定狀態不會隨新執行個體恢復，記憶體內的切換狀態就沒有意義。
<!-- ML4T_ZH_END c0037-p000 -->


In [17]:
print("\n4. Checking state persistence...")
safe_broker.close_persistence()
new_safe_broker = SafeBroker(MockBroker(), config)
print(f"   Kill switch still active: {new_safe_broker._state.kill_switch_activated}")
print(f"   Reason: {new_safe_broker._state.kill_switch_reason}")
assert new_safe_broker._state.kill_switch_activated

print("\n5. Disabling kill switch (manual recovery):")
new_safe_broker.disable_kill_switch()
print("   Kill switch disabled!")

print("\n6. Trading after recovery:")
new_safe_broker.record_market_snapshot("AAPL", 100.0)
_ = run_demo(new_safe_broker.submit_order_async("AAPL", 10, OrderSide.BUY))
new_safe_broker.close_persistence()


4. Checking state persistence...
2026-09-08 21:32:01,111 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: paper


2026-09-08 21:32:01,111 - ml4t.live.safety - WARNING - Kill switch was previously activated: Manual test - simulating emergency


   Kill switch still active: True
   Reason: Manual test - simulating emergency

5. Disabling kill switch (manual recovery):
2026-09-08 21:32:01,111 - ml4t.live.safety - WARNING - Kill switch DISABLED - proceed with caution!


   Kill switch disabled!

6. Trading after recovery:


2026-09-08 21:32:01,153 - ml4t.live.safety - INFO - SafeBroker: Submitting buy 10.0 AAPL


2026-09-08 21:32:01,154 - __main__ - INFO - MockBroker: buy 10.0 AAPL @ $100.00


   [OK] Order accepted by control layer


The second `SafeBroker` reads the latch from the state file, so the halt outlives the process
that set it rather than disappearing with the kernel. A halt that did not outlive its process
would let a restart quietly reactivate a strategy that was stopped for a real risk event.

<!-- ML4T_ZH_BEGIN c0039-p000 -->
第二個 `SafeBroker` 從狀態檔讀取鎖定狀態，因此停止狀態會延續到設定它的處理程序結束之後，而不會隨核心程序消失。若停止狀態無法跨程序保留，重新啟動就可能悄悄重啟因真實風險事件而停止的策略。
<!-- ML4T_ZH_END c0039-p000 -->


## 8. Demonstration: Shadow Mode

Shadow mode logs orders but doesn't execute them. Uses VirtualPortfolio
for realistic position tracking to prevent infinite buy loops.

<!-- ML4T_ZH_BEGIN c0040-p000 -->
## 8. 示範：影子模式

影子模式會記錄訂單，但不執行訂單。它使用 VirtualPortfolio 追蹤逼真的部位狀態，防止無限買入循環。
<!-- ML4T_ZH_END c0040-p000 -->


In [18]:
print("\n" + "=" * 70)
print("DEMO 6: Shadow Mode (No Broker Submission)")
print("=" * 70)

state_file = _temporary_state_path()
broker = MockBroker()
run_async(broker.connect())

config = LiveRiskConfig(
    shadow_mode=True,  # Enable shadow mode
    max_order_value=50_000.0,
    max_position_value=100_000.0,
    state_file=state_file,
)
safe_broker = SafeBroker(broker, config)
safe_broker.record_market_snapshot("AAPL", 100.0)

print("\n1. Submitting order in shadow mode:")
order = run_demo(safe_broker.submit_order_async("AAPL", 100, OrderSide.BUY))

print("\n2. Checking virtual position (shadows real broker):")
pos = safe_broker.get_position("AAPL")
if pos:
    print(f"   Virtual position: {pos.quantity} shares @ ${pos.entry_price:.2f}")
else:
    raise AssertionError("Shadow order did not update the virtual position")

print("\n3. Checking real broker position (should be empty):")
real_pos = broker.get_position("AAPL")
if real_pos:
    raise AssertionError(f"Shadow order reached the real broker: {real_pos.quantity} shares")
else:
    print("   Real broker has NO position (correct - shadow mode)")

print("\n4. Virtual portfolio account value:")
value = run_async(safe_broker.get_account_value_async())
print(f"   Virtual account value: ${value:,.2f}")


DEMO 6: Shadow Mode (No Broker Submission)
2026-09-08 21:32:01,202 - __main__ - INFO - MockBroker: Connected


2026-09-08 21:32:01,213 - ml4t.live.safety - INFO - SafeBroker initialized. Execution mode: shadow



1. Submitting order in shadow mode:


2026-09-08 21:32:01,235 - ml4t.live.safety - INFO - Shadow: Opened AAPL LONG 100.0


2026-09-08 21:32:01,235 - ml4t.live.safety - INFO - SHADOW: buy 100.0 AAPL @ $100.00 (value: $10,000)


   [OK] Order accepted by control layer

2. Checking virtual position (shadows real broker):
   Virtual position: 100.0 shares @ $100.00

3. Checking real broker position (should be empty):
   Real broker has NO position (correct - shadow mode)

4. Virtual portfolio account value:
   Virtual account value: $100,000.00


Shadow mode updates the virtual portfolio while the underlying broker stays flat, so the whole
routing path is observable without any change to real inventory. That is what makes it the
first place to validate end-to-end routing, ahead of paper and well ahead of live.

<!-- ML4T_ZH_BEGIN c0042-p000 -->
影子模式會更新虛擬投資組合，同時讓底層券商維持空部位，因此可以觀察整條訂單路由路徑，而不改變實際庫存。這使它成為驗證端對端路由的第一站，早於模擬交易，更早於實盤交易。
<!-- ML4T_ZH_END c0042-p000 -->


## 9. Demonstration: VirtualPortfolio Details

VirtualPortfolio handles position tracking for shadow mode,
including weighted average cost basis and position flipping.

<!-- ML4T_ZH_BEGIN c0043-p000 -->
## 9. 示範：VirtualPortfolio 細節

VirtualPortfolio 處理影子模式的部位追蹤，包括加權平均成本基礎與部位翻轉。
<!-- ML4T_ZH_END c0043-p000 -->


In [19]:
print("\n" + "=" * 70)
print("DEMO 7: VirtualPortfolio Position Tracking")
print("=" * 70)

portfolio = VirtualPortfolio(initial_cash=100_000.0)

print("\n1. Initial state:")
print(f"   Cash: ${portfolio.cash:,.2f}")
print(f"   Positions: {len(portfolio.positions)}")

# Simulate buy order
buy_order = Order(
    asset="AAPL",
    side=OrderSide.BUY,
    quantity=100,
    filled_quantity=100,
    filled_price=150.0,
    status=OrderStatus.FILLED,
)
portfolio.process_fill(buy_order)

print("\n2. After buying 100 AAPL @ $150:")
pos = portfolio.positions.get("AAPL")
print(f"   Position: {pos.quantity} shares @ ${pos.entry_price:.2f}")
print(f"   Cash: ${portfolio.cash:,.2f}")
print(f"   Account value: ${portfolio.account_value:,.2f}")
assert pos.quantity == 100
assert pos.entry_price == 150.0
assert portfolio.account_value == 100_000.0


DEMO 7: VirtualPortfolio Position Tracking

1. Initial state:
   Cash: $100,000.00
   Positions: 0
2026-09-08 21:32:01,286 - ml4t.live.safety - INFO - Shadow: Opened AAPL LONG 100



2. After buying 100 AAPL @ $150:
   Position: 100 shares @ $150.00
   Cash: $85,000.00
   Account value: $100,000.00


In [20]:
# Add to position at higher price
buy_order2 = Order(
    asset="AAPL",
    side=OrderSide.BUY,
    quantity=100,
    filled_quantity=100,
    filled_price=160.0,
    status=OrderStatus.FILLED,
)
portfolio.process_fill(buy_order2)

print("\n3. After buying 100 more AAPL @ $160 (weighted avg cost):")
pos = portfolio.positions.get("AAPL")
print(f"   Position: {pos.quantity} shares @ ${pos.entry_price:.2f}")
print(f"   Cash: ${portfolio.cash:,.2f}")
assert pos.quantity == 200
assert pos.entry_price == 155.0

# Partial sell
sell_order = Order(
    asset="AAPL",
    side=OrderSide.SELL,
    quantity=50,
    filled_quantity=50,
    filled_price=170.0,
    status=OrderStatus.FILLED,
)
portfolio.process_fill(sell_order)

print("\n4. After selling 50 AAPL @ $170:")
pos = portfolio.positions.get("AAPL")
print(f"   Position: {pos.quantity} shares @ ${pos.entry_price:.2f}")
print(f"   Cash: ${portfolio.cash:,.2f}")
print(f"   Account value: ${portfolio.account_value:,.2f}")
assert pos.quantity == 150
assert pos.entry_price == 155.0
assert portfolio.account_value == 103_000.0

2026-09-08 21:32:01,295 - ml4t.live.safety - INFO - Shadow: Increased AAPL to 200, basis $155.00



3. After buying 100 more AAPL @ $160 (weighted avg cost):
   Position: 200 shares @ $155.00
   Cash: $69,000.00
2026-09-08 21:32:01,295 - ml4t.live.safety - INFO - Shadow: Reduced AAPL to 150



4. After selling 50 AAPL @ $170:
   Position: 150 shares @ $155.00
   Cash: $77,500.00
   Account value: $103,000.00


In [21]:
for managed_path in MANAGED_STATE_PATHS:
    for candidate in (
        managed_path,
        managed_path.with_name(f"{managed_path.name}.lock"),
        managed_path.with_name(f"{managed_path.name}.head"),
    ):
        candidate.unlink(missing_ok=True)

unexpected_state_files = list(STATE_DIR.iterdir())
assert not unexpected_state_files, f"Temporary state residue: {unexpected_state_files}"
STATE_DIR.rmdir()
print(f"[OK] Cleaned {len(MANAGED_STATE_PATHS)} managed state and journal paths")

[OK] Cleaned 18 managed state and journal paths


The virtual portfolio carries cost basis and partial exits explicitly rather than leaving them
to an implementation detail. A shadow or paper environment without that accounting hides the
state-management bugs it exists to surface, and those are the bugs that cost money live.

<!-- ML4T_ZH_BEGIN c0047-p000 -->
虛擬投資組合會明確記錄成本基礎與部分平倉，而非將這些行為留給實作細節。若影子或模擬環境缺乏這種入帳機制，就會掩蓋其原本應揭露的狀態管理錯誤，而這些錯誤在實盤交易中會造成資金損失。
<!-- ML4T_ZH_END c0047-p000 -->


## Summary: Controls Exercised Above

| Control | What this notebook showed |
|---------|---------------------------|
| **Order Size Limits** | Max shares and value per order rejected before the broker sees them |
| **Position Limits** | Per-position value + share caps blocked unsafe accumulation |
| **Rate Limiting** | Burst of valid orders blocked at the per-minute cap |
| **Asset Restrictions** | Allowed / blocked asset lists held the universe boundary |
| **Kill Switch** | Manual halt, latch persisted across `SafeBroker` reconstruction |
| **Shadow Mode** | `VirtualPortfolio` tracked fills while the underlying broker stayed flat |

**Controls covered elsewhere**:
- **Duplicate Order Filter** and **Price Deviation** are configured through
  `LiveRiskConfig.dedup_window_seconds` and `LiveRiskConfig.max_price_deviation_pct`;
  this notebook does not present them as executed results.
- **Daily-Loss Drawdown Monitor** + **Stale-Data Rejection** are demonstrated in
  `13_runtime_safety_showcase`, which drives the same `SafeBroker` instance
  through both failure modes with full kill-switch latch.

**Additional features the demos rely on**:
- **State Persistence**: kill switch and daily counters survive restarts.
- **Atomic Writes**: the state file uses atomic JSON writes to prevent corruption.

**Best Practices**:
1. Always start with `shadow_mode=True`.
2. Graduate to paper trading.
3. Use conservative limits when going live.
4. Monitor the state file for kill-switch activations.

<!-- ML4T_ZH_BEGIN c0048-p000 -->
## 摘要：上述已測試的控制措施

| 控制措施 | 本 Notebook 呈現的結果 |
|---------|---------------------------|
| **訂單規模限額** | 超過每筆股數及金額上限的訂單，會在券商收到前遭拒絕 |
| **部位限額** | 個別部位金額及股數上限，會阻止不安全的累積 |
| **速率限制** | 有效訂單的突發流量在每分鐘上限處遭封鎖 |
| **資產限制** | 允許與封鎖資產清單維持標的範圍界線 |
| **Kill Switch** | 手動停止後，透過 `SafeBroker` 重建仍保留鎖定狀態 |
| **影子模式** | 底層券商維持空部位時，`VirtualPortfolio` 仍會追蹤成交 |

**其他地方涵蓋的控制措施**：
- **重複訂單過濾**與**價格偏離檢查**透過 `LiveRiskConfig.dedup_window_seconds` 與 `LiveRiskConfig.max_price_deviation_pct` 設定；本 Notebook 不將其呈現為已執行的結果。
- **每日損失回撤監控**與**過時資料拒絕**在 `13_runtime_safety_showcase` 示範；該示範會讓同一個 `SafeBroker` 執行個體經歷兩種失敗模式，並完整保留 Kill Switch 鎖定狀態。

**示範所依賴的其他功能**：
- **狀態持久化**：Kill Switch 與每日計數器在重啟後仍會保留。
- **原子寫入**：狀態檔採用原子方式寫入 JSON，以防止損毀。

**最佳實務**：
1. 一律先從 `shadow_mode=True` 開始。
2. 再進入模擬交易。
3. 進行實盤交易時使用保守的限額。
4. 監控狀態檔中的 Kill Switch 啟動紀錄。
<!-- ML4T_ZH_END c0048-p000 -->


## Key Takeaways

1. **Defense-in-depth**: SafeBroker layers independent controls so that no
   single misconfiguration can produce an unchecked order. Six of those
   controls are exercised above; the remaining duplicate-order, fat-finger,
   and drawdown gates are wired through the same `LiveRiskConfig` surface
   without being claimed as outputs of this notebook.
2. **Configurable via LiveRiskConfig**: order size, position exposure, rate
   caps and drawdown thresholds are all parameters rather than hard-coded
   logic.
3. **Kill switch persists across restarts**: Emergency halts survive process
   restarts and must be manually cleared, preventing accidental
   reactivation of a halted strategy.
4. **Shadow mode with VirtualPortfolio**: Orders are logged and tracked with
   realistic cost-basis accounting without touching the broker, making it
   the safest first deployment step.

**Next**: Combine these controls with the parity checks in `08_pipeline_verification` and then keep the
same SafeBroker configuration when moving from shadow mode to paper trading.

<!-- ML4T_ZH_BEGIN c0049-p000 -->
## 重點整理

1. **多層防護**：SafeBroker 疊加獨立控制措施，避免單一設定錯誤產生未經檢查的訂單。上述示範測試其中六項；其餘重複訂單、輸入失誤與回撤限制也透過同一個 `LiveRiskConfig` 介面接入，但不宣稱它們是本 Notebook 的執行結果。
2. **可透過 LiveRiskConfig 設定**：訂單規模、部位曝險、速率上限與回撤門檻都是參數，而非寫死的邏輯。
3. **Kill Switch 可跨重啟保留**：緊急停止狀態會在處理程序重啟後維持，且必須手動解除，以防停止的策略意外重新啟動。
4. **使用 VirtualPortfolio 的影子模式**：訂單會以逼真的成本基礎入帳方式記錄與追蹤，但不觸及券商，因此是最安全的初始部署步驟。

**接下來**：將這些控制措施與 `08_pipeline_verification` 的一致性檢查結合，從影子模式轉至模擬交易時沿用相同的 SafeBroker 設定。
<!-- ML4T_ZH_END c0049-p000 -->
